In [1]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import os
os.environ["USER"] = "aleksa"
os.environ["HF_HOME"] = "/data"
!nvidia-smi

Wed Aug 12 20:27:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.10              Driver Version: 570.86.10      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:90:00.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!which python

/opt/conda/bin/python


In [4]:
!pip install accelerate

import sys
sys.path.append("/home/mls07/speculative-decoding/src")
from model import load_tokenizer, load_teacher, load_student

tokenizer = load_tokenizer()

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
!nvidia-smi
!ps aux | grep python
!ps aux | grep jupyter

0.0 GB allocated
0.0 GB reserved
Wed Aug 12 20:27:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.10              Driver Version: 570.86.10      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:90:00.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------

In [6]:
teacher = load_teacher()

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

In [7]:
prompt = "あなたの名前は何ですか"

inputs = tokenizer(prompt, return_tensors="pt").to(teacher.device)

with torch.no_grad():
    output_ids = teacher.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

あなたの名前は何ですか。 A. たなかです。 B. タナカです。 C. ツナカです。 D. タナカさんです。

タナカです。

The new policy has not yet been ________ (批准). (根据中文提示填空)

approved

He is an ________ (有经验的) teacher. (根据中文提示填空)

experienced

We are ________ (在学习) Chinese. (根据中文提示填空)

learning




In [8]:
print(teacher.device)

cuda:0


In [9]:
student = load_student()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [10]:
with torch.no_grad():
    output_ids = student.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

あなたの名前は何ですか？
A. 木村
B. 小野
C. 木村太郎
D. 小野太郎
答案:
C

在进行单相电能表现场检验时，要求电能表的误差在允许范围以内，具体误差为（ ）以内。此外，电能表的误差需要满足负载功率因数为0.85时的误差限值。请问，电能表的误差应控制在哪个范围？



In [11]:
with torch.no_grad():
    output_ids = teacher.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
    )
output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output)

あなたの名前は何ですか。 ______は木村太郎です。 木村太郎

关于企业价值评估，下列说法不正确的有( )。 A.实体价值=股权价值+债务价值 B.股权价值=股票数量×每股价值=整体价值－付息债务价值 C.为了计算股权价值，必须明确股东权益的内涵 D.一个企业的实体价值应是各单项资产价值的总和 A. B. C. D. AD

请听录音，填写下面的空白处的单词或短语（10选6）（6分） 汉语学习法_1_是汉语学习的开始，它是一门系统地、专门地、科学地讲授______发音规律和______学习的方法的课。汉语语音学、______的拼音方案、______系统和汉语的词汇、语音和句法中所特有的基本概念都是这门课程的重要内容。汉语学习的第一个障碍是汉语的语音
